# ASSIGNMENT 7

## Creating and Fixing Mode Collapse in a GAN

### Objective

The purpose of this assignment is to understand **mode collapse** in Generative Adversarial Networks by intentionally making a DCGAN unstable and then applying one technique to improve its behavior.

The experiment has two stages:

1. Train a deliberately unstable GAN and observe whether its generated samples become very similar.
2. Apply **label smoothing** and retrain the model, then compare the generated outputs.


## 1. Required Libraries

The following libraries are used to build and train the DCGAN, load the MNIST dataset, visualize generated images, and save the resulting image grids.


In [1]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

os.makedirs("assignment7_outputs", exist_ok=True)


Using device: cpu


## 2. Load the MNIST Dataset

MNIST contains grayscale handwritten digits. The images are resized to the 28 × 28 format and normalized to the range expected by the generator's `tanh` output.


In [2]:
BATCH_SIZE = 128
IMAGE_SIZE = 28
CHANNELS = 1
LATENT_DIM = 100
EPOCHS = 15

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

print("Training images:", len(train_dataset))


100%|██████████| 9.91M/9.91M [00:01<00:00, 4.97MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 125kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.22MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 4.87MB/s]


Training images: 60000


## 3. DCGAN Model

The generator converts a random latent vector into a 28 × 28 digit image. The discriminator receives an image and predicts whether it is real or generated.

For this assignment, the model definition supports both the deliberately unstable version and the improved version. Batch normalization can be disabled in the generator to intentionally reduce training stability.


In [3]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, use_batchnorm=True, base_channels=64):
        super().__init__()

        layers = [
            nn.ConvTranspose2d(latent_dim, base_channels * 4, 7, 1, 0, bias=not use_batchnorm)
        ]

        if use_batchnorm:
            layers.append(nn.BatchNorm2d(base_channels * 4))

        layers.append(nn.ReLU(True))

        layers.extend([
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, 4, 2, 1, bias=not use_batchnorm)
        ])

        if use_batchnorm:
            layers.append(nn.BatchNorm2d(base_channels * 2))

        layers.append(nn.ReLU(True))
        layers.append(
            nn.ConvTranspose2d(base_channels * 2, CHANNELS, 4, 2, 1)
        )
        layers.append(nn.Tanh())

        self.model = nn.Sequential(*layers)

    def forward(self, z):
        return self.model(z)


class Discriminator(nn.Module):
    def __init__(self, base_channels=64):
        super().__init__()

        self.model = nn.Sequential(
            nn.Conv2d(CHANNELS, base_channels, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(base_channels, base_channels * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(base_channels * 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(base_channels * 2, base_channels * 4, 3, 2, 1, bias=False),
            nn.BatchNorm2d(base_channels * 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Flatten(),
            nn.Linear(base_channels * 4 * 4 * 4, 1)
        )

    def forward(self, x):
        return self.model(x)


def initialize_weights(model):
    for layer in model.modules():
        if isinstance(layer, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
            nn.init.normal_(layer.weight, 0.0, 0.02)
            if layer.bias is not None:
                nn.init.constant_(layer.bias, 0)
        elif isinstance(layer, nn.BatchNorm2d):
            nn.init.normal_(layer.weight, 1.0, 0.02)
            nn.init.constant_(layer.bias, 0)


def build_models(use_batchnorm=True, generator_channels=64):
    G = Generator(
        latent_dim=LATENT_DIM,
        use_batchnorm=use_batchnorm,
        base_channels=generator_channels
    ).to(device)

    D = Discriminator(base_channels=64).to(device)

    initialize_weights(G)
    initialize_weights(D)

    return G, D


## 4. Helper Function for Visualizing Generated Images

The same fixed latent vectors are used before and after the improvement. This makes the visual comparison easier because the models are asked to generate images from identical random inputs.


In [4]:
FIXED_NOISE = torch.randn(64, LATENT_DIM, 1, 1, device=device)

def show_generated_images(generator, title, filename):
    generator.eval()

    with torch.no_grad():
        fake = generator(FIXED_NOISE).cpu()

    # Convert from [-1, 1] to [0, 1] for display
    fake = (fake + 1) / 2

    grid = make_grid(fake, nrow=8, padding=2)

    plt.figure(figsize=(8, 8))
    plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
    plt.title(title)
    plt.axis("off")
    plt.show()

    save_image(fake, filename, nrow=8)

    generator.train()
    return fake


# Part A — Intentionally Create an Unstable GAN

## 5. Modification Used to Produce Poor Training

To deliberately make the GAN harder to train, the generator is changed in three ways:

- Batch normalization is removed from the generator.
- The generator capacity is reduced.
- A significantly larger learning rate is used.

These changes are intended to make the generator-discriminator training balance less stable and increase the possibility of repetitive outputs.


In [5]:
# Deliberately unstable configuration
BROKEN_LR = 0.005
BROKEN_G_CHANNELS = 32

broken_G, broken_D = build_models(
    use_batchnorm=False,
    generator_channels=BROKEN_G_CHANNELS
)

criterion = nn.BCEWithLogitsLoss()

optimizer_G = optim.Adam(
    broken_G.parameters(),
    lr=BROKEN_LR,
    betas=(0.5, 0.999)
)

optimizer_D = optim.Adam(
    broken_D.parameters(),
    lr=BROKEN_LR,
    betas=(0.5, 0.999)
)

print("Broken GAN configuration:")
print("Generator batch normalization: disabled")
print("Generator base channels:", BROKEN_G_CHANNELS)
print("Learning rate:", BROKEN_LR)


Broken GAN configuration:
Generator batch normalization: disabled
Generator base channels: 32
Learning rate: 0.005


In [6]:
def train_gan(
    generator,
    discriminator,
    loader,
    optimizer_g,
    optimizer_d,
    epochs,
    label_smoothing=False,
    show_progress=True
):
    g_losses = []
    d_losses = []

    for epoch in range(epochs):
        generator.train()
        discriminator.train()

        running_g = 0.0
        running_d = 0.0

        for real_images, _ in loader:
            real_images = real_images.to(device)
            batch_size = real_images.size(0)

            # -------------------------
            # Train discriminator
            # -------------------------
            optimizer_d.zero_grad()

            real_logits = discriminator(real_images)

            if label_smoothing:
                real_targets = torch.full_like(real_logits, 0.9)
            else:
                real_targets = torch.ones_like(real_logits)

            real_loss = criterion(real_logits, real_targets)

            noise = torch.randn(batch_size, LATENT_DIM, 1, 1, device=device)
            fake_images = generator(noise)

            fake_logits = discriminator(fake_images.detach())
            fake_targets = torch.zeros_like(fake_logits)

            fake_loss = criterion(fake_logits, fake_targets)

            d_loss = real_loss + fake_loss
            d_loss.backward()
            optimizer_d.step()

            # -------------------------
            # Train generator
            # -------------------------
            optimizer_g.zero_grad()

            noise = torch.randn(batch_size, LATENT_DIM, 1, 1, device=device)
            generated = generator(noise)
            generated_logits = discriminator(generated)

            # Generator wants the discriminator to classify fake images as real.
            if label_smoothing:
                generator_targets = torch.full_like(generated_logits, 0.9)
            else:
                generator_targets = torch.ones_like(generated_logits)

            g_loss = criterion(generated_logits, generator_targets)

            g_loss.backward()
            optimizer_g.step()

            running_d += d_loss.item()
            running_g += g_loss.item()

        avg_d = running_d / len(loader)
        avg_g = running_g / len(loader)

        d_losses.append(avg_d)
        g_losses.append(avg_g)

        if show_progress:
            print(
                f"Epoch [{epoch+1:02d}/{epochs}] "
                f"D Loss: {avg_d:.4f} | G Loss: {avg_g:.4f}"
            )

    return g_losses, d_losses


## 6. Train the Unstable Model

The modified GAN is trained for 15 epochs. After training, the generated images are displayed and saved so that the output can be inspected for signs of mode collapse, such as repeated or highly similar digits.


In [ ]:
broken_g_losses, broken_d_losses = train_gan(
    broken_G,
    broken_D,
    train_loader,
    optimizer_G,
    optimizer_D,
    epochs=EPOCHS,
    label_smoothing=False
)

broken_images = show_generated_images(
    broken_G,
    "Broken / Unstable GAN Output",
    "assignment7_outputs/broken_output.png"
)


### Interpretation of the Broken Output

If mode collapse occurs, several generated samples may look almost identical or may repeatedly represent only a small number of digit patterns. This happens because the generator can discover a limited set of outputs that successfully fool the discriminator instead of learning the full diversity present in the training data.

**Submission screenshot:** The image grid generated by the previous cell can be used as the screenshot of the broken/collapsed output.


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(broken_g_losses, label="Generator Loss")
plt.plot(broken_d_losses, label="Discriminator Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Losses — Unstable GAN")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


# Part B — Fixing the Problem

## 7. Improvement Technique: Label Smoothing

For the improved model, **one technique—label smoothing—is applied**.

Instead of training the discriminator with an exact real label of 1.0, the real target is changed to 0.9. This prevents the discriminator from becoming excessively confident about real samples and can provide more useful gradients to the generator.

The improved experiment restores batch normalization and the original generator capacity as well, while keeping the main comparison focused on the label-smoothing technique.


In [ ]:
# Improved configuration
FIXED_LR = 0.0002
FIXED_G_CHANNELS = 64

fixed_G, fixed_D = build_models(
    use_batchnorm=True,
    generator_channels=FIXED_G_CHANNELS
)

fixed_optimizer_G = optim.Adam(
    fixed_G.parameters(),
    lr=FIXED_LR,
    betas=(0.5, 0.999)
)

fixed_optimizer_D = optim.Adam(
    fixed_D.parameters(),
    lr=FIXED_LR,
    betas=(0.5, 0.999)
)

print("Improved GAN configuration:")
print("Generator batch normalization: enabled")
print("Generator base channels:", FIXED_G_CHANNELS)
print("Learning rate:", FIXED_LR)
print("Real-label smoothing value: 0.9")


## 8. Retrain the Improved GAN

The improved configuration is trained for the same 15 epochs. The fixed latent vectors are then used to produce an output grid for comparison with the unstable model.


In [ ]:
fixed_g_losses, fixed_d_losses = train_gan(
    fixed_G,
    fixed_D,
    train_loader,
    fixed_optimizer_G,
    fixed_optimizer_D,
    epochs=EPOCHS,
    label_smoothing=True
)

fixed_images = show_generated_images(
    fixed_G,
    "Improved GAN Output — Label Smoothing",
    "assignment7_outputs/improved_output.png"
)


### Improved Output

The improved output should be compared with the broken output using the same fixed latent vectors. A successful improvement should generally show greater variation between samples and more recognizable digit structures.

**Submission screenshot:** The image grid generated by the previous cell can be used as the screenshot of the improved output.


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(fixed_g_losses, label="Generator Loss")
plt.plot(fixed_d_losses, label="Discriminator Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Losses — Improved GAN")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## 9. Side-by-Side Comparison

The following visualization compares the generated samples from the unstable and improved models using the same latent vectors.


In [ ]:
comparison = torch.cat([broken_images[:32], fixed_images[:32]], dim=0)

plt.figure(figsize=(10, 10))
grid = make_grid(comparison, nrow=8, padding=2)
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.title("Top: Broken GAN | Bottom: Improved GAN")
plt.axis("off")
plt.show()


## Conclusion

This experiment demonstrates that GAN training is sensitive to the balance between the generator and discriminator. Deliberately reducing generator capacity, removing normalization, and using an excessively high learning rate can make training unstable and encourage mode collapse. Applying label smoothing and returning to a more stable training configuration can improve the quality and diversity of generated samples.